# 01. CarDD → YOLO Segmentation (full baseline)

Этот ноутбук **не скачивает** датасеты.  
Он берёт уже распакованный `CarDD` и собирает baseline YOLO-seg датасет в `ml/data/cardd_yolo_seg_full`.

Здесь сохраняются **все 6 классов повреждений**, используются исходные split'ы CarDD и весь набор позитивных изображений без undersampling и без negative-примеров.

In [ ]:

!pip install -q opencv-python-headless pillow numpy pyyaml tqdm pycocotools

In [ ]:

import os
import sys
import json
import shutil
from pathlib import Path
from collections import defaultdict

import cv2
import yaml
import numpy as np
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if any((cand / marker).exists() for marker in ["pyproject.toml", "requirements.txt", ".git"]):
            return cand
    return start.parent if start.parent.exists() else start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.damage_seg.dataset_builder import build_full_cardd_damage_seg_dataset

In [ ]:

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if any((cand / m).exists() for m in ["pyproject.toml", "requirements.txt", ".git"]):
            return cand
    return start.parent if start.parent.exists() else start

PROJECT_ROOT = find_project_root(Path("."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "ml" / "data"
CARDD_DIR = DATA_ROOT / "CarDD"
YOLO_DATASET_DIR = DATA_ROOT / "cardd_yolo_seg_full"
YOLO_DATASET_DIR.mkdir(parents=True, exist_ok=True)

DAMAGE_CLASSES = ["dent", "scratch", "crack", "glass_shatter", "lamp_broken", "tire_flat"]
COCO_TO_YOLO_CLASS_MAP = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
    5: 4,
    6: 5,
}
SPLIT_MAP = {
    "instances_train2017.json": ("train2017", "train"),
    "instances_val2017.json": ("val2017", "val"),
    "instances_test2017.json": ("test2017", "test"),
}
(PROJECT_ROOT / "ml" / "damage_seg" / "configs").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "ml" / "damage_seg" / "configs" / "cardd_seg_dataset_build_full.json").write_text(
    json.dumps({"mode": "full_cardd_baseline"}, indent=2),
    encoding="utf-8",
)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("CARDD_DIR    =", CARDD_DIR)
print("YOLO_DATASET_DIR =", YOLO_DATASET_DIR)

In [ ]:

def find_cardd_coco_dir(cardd_dir: Path):
    candidates = [
        cardd_dir / "CarDD_release" / "CarDD_COCO",
        cardd_dir / "CarDD_COCO",
        cardd_dir,
    ]
    for c in candidates:
        if (c / "annotations").exists():
            return c
    raise FileNotFoundError("Не удалось найти папку CarDD_COCO/annotations")

def normalize_polygon(poly, width, height):
    pts = []
    for i in range(0, len(poly), 2):
        x = min(max(poly[i] / width, 0.0), 1.0)
        y = min(max(poly[i+1] / height, 0.0), 1.0)
        pts.extend([x, y])
    return pts

def rle_to_polygons(segmentation, height, width):
    mask = mask_utils.decode(segmentation)
    if mask.ndim == 3:
        mask = np.any(mask, axis=2).astype(np.uint8)
    else:
        mask = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        cnt = cnt.squeeze(axis=1)
        if cnt.ndim != 2 or len(cnt) < 3:
            continue
        poly = []
        for x, y in cnt:
            poly.extend([float(x), float(y)])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

def ann_to_polygons(ann, height, width):
    seg = ann.get("segmentation", [])
    if not seg:
        return []
    if isinstance(seg, dict):
        return rle_to_polygons(seg, height, width)
    polygons = []
    for poly in seg:
        if len(poly) >= 6:
            polygons.append(poly)
    return polygons

In [ ]:

def prepare_cardd_coco_to_yolo(cardd_dir: Path, output_dir: Path, clear_existing: bool = True):
    coco_dir = find_cardd_coco_dir(cardd_dir)
    print("Found CarDD COCO at:", coco_dir)

    if clear_existing and output_dir.exists():
        for subdir in ["images", "labels"]:
            old = output_dir / subdir
            if old.exists():
                shutil.rmtree(old)

    total_images, total_objects = 0, 0

    for ann_filename, (img_dir_name, split_name) in SPLIT_MAP.items():
        ann_path = coco_dir / "annotations" / ann_filename
        img_dir = coco_dir / img_dir_name
        if not ann_path.exists() or not img_dir.exists():
            print(f"[WARN] missing split resources for {split_name}")
            continue

        out_images = output_dir / "images" / split_name
        out_labels = output_dir / "labels" / split_name
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        with open(ann_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        images_by_id = {img["id"]: img for img in coco["images"]}
        anns_by_img = defaultdict(list)
        for ann in coco["annotations"]:
            anns_by_img[ann["image_id"]].append(ann)

        split_images, split_objects = 0, 0
        for img_id, img_info in tqdm(images_by_id.items(), desc=f"Convert {split_name}"):
            filename = img_info["file_name"]
            width = img_info["width"]
            height = img_info["height"]

            src_img = img_dir / filename
            if not src_img.exists():
                continue

            shutil.copy2(src_img, out_images / filename)

            lines = []
            for ann in anns_by_img.get(img_id, []):
                cat_id = ann.get("category_id")
                if cat_id not in COCO_TO_YOLO_CLASS_MAP:
                    continue
                cls_idx = COCO_TO_YOLO_CLASS_MAP[cat_id]
                polys = ann_to_polygons(ann, height, width)
                for poly in polys:
                    norm = normalize_polygon(poly, width, height)
                    if len(norm) >= 6:
                        lines.append(f"{cls_idx} " + " ".join(f"{v:.6f}" for v in norm))
                        split_objects += 1

            label_path = out_labels / f"{Path(filename).stem}.txt"
            with open(label_path, "w", encoding="utf-8") as f:
                f.write("\n".join(lines))
            split_images += 1

        print(f"[{split_name}] images={split_images} objects={split_objects}")
        total_images += split_images
        total_objects += split_objects

    data_yaml = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(DAMAGE_CLASSES)},
    }
    with open(output_dir / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    print("Done. total_images =", total_images, " total_objects =", total_objects)
    print("data.yaml ->", output_dir / "data.yaml")
    return output_dir

In [ ]:

summary = build_full_cardd_damage_seg_dataset(
    cardd_dir=CARDD_DIR,
    output_dir=YOLO_DATASET_DIR,
    clear_existing=True,
    seed=42,
)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("YOLO dataset ready at:", YOLO_DATASET_DIR)